In [ ]:
# ---- 1. setup ---------------------------------------------------------------------
# in the beginning everything was written to /content/outputs and the splits
# disappeared when the session ended, so now the outputs go to Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("drive mount failed:", e)   # this happens when the cell is re-run

import os

DATA = '/content/data'                         # the kaggle zip goes here
OUT  = '/content/drive/MyDrive/MDL_HW1'        # figures / tables / checkpoints
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

if not os.path.exists(DATA + '/HAM10000_metadata.csv'):
    print("downloading from Kaggle...")
    # TODO: move the token to a colab secret at some point. left here for now
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_9d722ed3fcd37402188b3ec9c53f0f99'
    !pip install -q kaggle --upgrade
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data --unzip
    print("dataset downloaded")
else:
    print("dataset already in /content/data")

!ls /content/data

In [ ]:
# ---- 2. dataset + split -----------------------------------------------------------
# HAM10000 has 10015 images but only 7470 lesions: the same lesion was
# photographed up to 4 times (different angles / body parts).
#
# v1 of this cell used train_test_split(stratify=dx) directly on the images,
# so the same lesion ended up in train and in test and the first results
# looked too good (0.83 acc for a CNN trained from scratch on 7k images is
# not believable). now the split is done on lesion_id and the images follow.
import pandas as pd
from sklearn.model_selection import train_test_split

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'mel', 'vasc']
SEED = 42

def make_grouped_split(meta, seed=SEED):
    """70/15/15 on lesion_id, stratified on the class of each lesion."""
    meta = meta.copy()
    meta['label']  = meta['dx'].map({c: i for i, c in enumerate(CLASSES)})
    meta['binary'] = (meta['dx'] == 'mel').astype(int)   # melanoma vs rest

    # one row per lesion, if a lesion has two dx values (rare) the first is used
    lesions = meta.groupby('lesion_id', as_index=False).first()[['lesion_id', 'label']]

    tr_l, tmp_l = train_test_split(lesions, test_size=0.30, stratify=lesions['label'], random_state=seed)
    va_l, te_l = train_test_split(tmp_l,  test_size=0.50, stratify=tmp_l['label'],     random_state=seed)

    tr = meta[meta['lesion_id'].isin(tr_l['lesion_id'])]
    va = meta[meta['lesion_id'].isin(va_l['lesion_id'])]
    te = meta[meta['lesion_id'].isin(te_l['lesion_id'])]
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

meta = pd.read_csv(os.path.join(DATA, 'HAM10000_metadata.csv'))
print("metadata:", meta.shape, " lesions:", meta['lesion_id'].nunique())

train_df, val_df, test_df = make_grouped_split(meta)

# sanity checks, printed on purpose (they go in the report, section 2.1)
overlap_tr_te = set(train_df['lesion_id']) & set(test_df['lesion_id'])
overlap_tr_va = set(train_df['lesion_id']) & set(val_df['lesion_id'])
print("overlap train/test lesions:", len(overlap_tr_te))
print("overlap train/val  lesions:", len(overlap_tr_va))
assert len(overlap_tr_te) == 0 and len(overlap_tr_va) == 0, "lesion leaked across splits!"

# pos_weight for the binary loss, computed from the train set only
pos = train_df['binary'].sum()
neg = len(train_df) - pos
POS_WEIGHT = float(neg) / float(pos)
print(f"pos_weight (neg/pos, train only) = {POS_WEIGHT:.3f}")

print("split sizes: train", len(train_df), " val", len(val_df), " test", len(test_df))
print()
print("images per class")
print("class    train  val  test")
for i, c in enumerate(CLASSES):
    a = int((train_df['label'] == i).sum())
    b = int((val_df['label'] == i).sum())
    d = int((test_df['label'] == i).sum())
    print(f"{c:<7} {a:5d} {b:4d} {d:5d}")

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(os.path.join(OUT, name + '.csv'), index=False)
print("splits saved to", OUT)

In [ ]:
# ---- 3. imports, seed, device -----------------------------------------------------
import os, random, time, json
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, confusion_matrix, roc_curve)

def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

MEAN = (0.485, 0.456, 0.406)     # imagenet statistics, the pretrained models expect them
STD  = (0.229, 0.224, 0.225)

In [ ]:
# ---- 4. dataset + dataloaders -----------------------------------------------------
# the kaggle zip has the images in two folders, we look in part_1 then part_2
IMG1 = os.path.join(DATA, 'HAM10000_images_part_1')
IMG2 = os.path.join(DATA, 'HAM10000_images_part_2')

def image_path(image_id):
    p1 = os.path.join(IMG1, image_id + '.jpg')
    return p1 if os.path.exists(p1) else os.path.join(IMG2, image_id + '.jpg')

# reload the splits that cell 1 wrote to Drive (the runtime may have restarted)
train_df = pd.read_csv(os.path.join(OUT, 'train.csv'))
val_df   = pd.read_csv(os.path.join(OUT, 'val.csv'))
test_df  = pd.read_csv(os.path.join(OUT, 'test.csv'))
for d in (train_df, val_df, test_df):
    d['path'] = d['image_id'].apply(image_path)

print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("missing image files:", (~train_df['path'].apply(os.path.exists)).sum())

In [ ]:
class HAM(Dataset):
    """returns (image tensor, label). label = 0..6 for multi, 0/1 for binary."""
    def __init__(self, df, transform, mode='multi'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        if self.mode == 'multi':
            y = int(row['label'])
        else:
            y = int(row['binary'])
        return img, y

def make_loaders(mode='multi', bs=32, use_sampler=True, size=224):
    """train loader is augmented + balanced, val/test are only resized."""
    train_tf = A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.ColorJitter(0.1, 0.1, 0.1, 0.05, p=0.5),
        # A.RandomBrightnessContrast(0.3, 0.3, p=1.0),  # too strong, washed out the dermoscopy
        A.Normalize(MEAN, STD),
        ToTensorV2(),
    ])
    eval_tf = A.Compose([A.Resize(height=size, width=size), A.Normalize(MEAN, STD), ToTensorV2()])

    tr_ds = HAM(train_df, train_tf, mode)
    va_ds = HAM(val_df,   eval_tf,  mode)
    te_ds = HAM(test_df,  eval_tf,  mode)

    if use_sampler:
        # the classes are very unbalanced (nv = 67%, df = 1%). without this the
        # model just answers 'nv' for everything and still gets 60% accuracy
        col = 'label' if mode == 'multi' else 'binary'
        counts = train_df[col].value_counts().sort_index().values
        w = 1.0 / counts
        sw = [w[k] for k in train_df[col].values]
        sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
        tr_loader = DataLoader(tr_ds, batch_size=bs, sampler=sampler, num_workers=2, pin_memory=True)
    else:
        tr_loader = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)

    va_loader = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, te_loader

trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)
xb, yb = next(iter(trl))
print("batch:", tuple(xb.shape), "labels:", tuple(yb.shape), "classes in batch:", sorted(yb.unique().tolist()))

In [ ]:
# ---- 5. custom CNN ----------------------------------------------------------------
# 5 stages, every stage = 2x conv3x3 + batchnorm, resolution halved between
# stages with maxpool, then global average pooling + one linear layer.
# 11.02M parameters (table 4 in the report).
class CustomCNN(nn.Module):
    def __init__(self, n_out=7, width=32, dropout=0.3):
        super().__init__()
        c1, c2, c3, c4, c5 = width, width*2, width*4, width*8, width*16
        self.features = nn.Sequential(
            # stage 1 -> 224
            nn.Conv2d(3, c1, 3, 1, 1, bias=False),  nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, 1, 1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            # stage 2 -> 112
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.Conv2d(c2, c2, 3, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            # stage 3 -> 56
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, 3, 1, 1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.Conv2d(c3, c3, 3, 1, 1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            # stage 4 -> 28
            nn.MaxPool2d(2),
            nn.Conv2d(c3, c4, 3, 1, 1, bias=False), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.Conv2d(c4, c4, 3, 1, 1, bias=False), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            # stage 5 -> 14
            nn.MaxPool2d(2),
            nn.Conv2d(c4, c5, 3, 1, 1, bias=False), nn.BatchNorm2d(c5), nn.ReLU(inplace=True),
            nn.Conv2d(c5, c5, 3, 1, 1, bias=False), nn.BatchNorm2d(c5), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(c5, n_out)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(self.drop(x))

_n = CustomCNN(7)
print("custom cnn params: %.2fM" % (sum(p.numel() for p in _n.parameters()) / 1e6))

In [ ]:
# ---- 6. evaluation function -------------------------------------------------------
def evaluate(model, loader, mode):
    """returns the metrics + the raw predictions (we need them for the tables)."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            if mode == 'multi':
                p = torch.softmax(out, dim=1).cpu().numpy()
            else:
                p = torch.sigmoid(out.view(-1)).cpu().numpy()
            ys.append(y.numpy())
            ps.append(p)
    y = np.concatenate(ys)
    p = np.concatenate(ps)

    if mode == 'multi':
        pred = p.argmax(1)
        acc = accuracy_score(y, pred)
        _, _, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
        return {'acc': acc, 'macro_f1': f1, 'y': y, 'p': p, 'pred': pred}
    else:
        pred = (p >= 0.5).astype(int)
        acc = accuracy_score(y, pred)
        auc = roc_auc_score(y, p)
        ap  = average_precision_score(y, p)
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
        return {'acc': acc, 'auc': auc, 'pr_auc': ap, 'precision': pr, 'recall': rc,
                'f1': f1, 'y': y, 'p': p, 'pred': pred}

print("evaluate ready")


def _print_class_report(res, title):
    """small helper, prints precision/recall/F1 per class for a multi model."""
    pr, rc, f1, sup = precision_recall_fscore_support(res['y'], res['pred'], labels=range(7), zero_division=0)
    print()
    print("per class -", title)
    print("class    prec   rec    f1    n")
    for i, c in enumerate(CLASSES):
        print(f"{c:<7} {pr[i]:.3f}  {rc[i]:.3f}  {f1[i]:.3f} {sup[i]:5d}")

In [ ]:
# ---- 8. train the custom cnn (7 classes) ------------------------------------------
# from scratch, 25 epochs max, early stop after 5 epochs without improvement.
# on a T4 one epoch is ~10 min with the sampler on the augmented train set.
EPOCHS = 25    # tried 30 first, it never got past ~18 because of early stop
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)

custom_multi = CustomCNN(n_out=7).to(device)
custom_multi, hist_custom_multi = train_model(custom_multi, trl, val_loader, 'multi',
                                             epochs=EPOCHS, lr=1e-3, tag='custom-7cls')

res = evaluate(custom_multi, te_loader, 'multi')
print()
print("TEST custom cnn (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'custom cnn / multi')

torch.save({'state_dict': custom_multi.state_dict(),
            'history': hist_custom_multi,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'custom_multi.pt'))
print("saved", os.path.join(OUT, 'custom_multi.pt'))

In [ ]:
# ---- 9. train the custom cnn (melanoma vs rest) -----------------------------------
# TODO: the sampler helps here too but it makes the loss very noisy, decide later
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=32)

custom_bin = CustomCNN(n_out=1).to(device)     # one logit -> BCEWithLogitsLoss
custom_bin, hist_custom_bin = train_model(custom_bin, trl_b, val_b, 'binary',
                                         epochs=EPOCHS, lr=1e-3, tag='custom-bin')

res_b = evaluate(custom_bin, te_b, 'binary')
print()
print("TEST custom cnn (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': custom_bin.state_dict(),
            'history': hist_custom_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'custom_binary.pt'))
print("saved", os.path.join(OUT, 'custom_binary.pt'))

In [ ]:
# ---- 10. timm (pretrained models) --------------------------------------------------
!pip install -q timm
import timm
print("timm", timm.__version__)

def build_pretrained(name, n_out):
    """imagenet weights + a new head with n_out outputs."""
    return timm.create_model(name, pretrained=True, num_classes=n_out)

def freeze_backbone(model):
    """only the classifier is trainable (stage 1 of the fine-tuning)."""
    for p in model.parameters():
        p.requires_grad = False
    for n, p in model.named_parameters():
        if n.startswith(('fc', 'classifier', 'head')):
            p.requires_grad = True

def unfreeze_last_third(model):
    """stage 2: the last ~1/3 of the layers, the first ones are too generic."""
    params = list(model.named_parameters())
    for n, p in params[int(len(params) * 2 / 3):]:
        p.requires_grad = True

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

def finetune(model, tag, tr_loader, va_loader, mode, epochs=(2, 4, 8), loss_fn=None):
    """3 stages: head only -> last third -> everything, with smaller lr each time."""
    hist = {'train_loss': [], 'val_score': []}
    stages = [(freeze_backbone, epochs[0], 1e-3),
              (unfreeze_last_third, epochs[1], 3e-4),
              (unfreeze_all, epochs[2], 1e-4)]
    for i, (unfreezer, n_ep, lr) in enumerate(stages):
        unfreezer(model)
        model, h = train_model(model, tr_loader, va_loader, mode,
                               epochs=n_ep, lr=lr, loss_fn=loss_fn, tag=f"{tag}-s{i+1}")
        hist['train_loss'] += h['train_loss']
        hist['val_score']  += h['val_score']
    return model, hist


print("timm helpers ready")

In [ ]:
# ---- 11. resnet101, 7 classes ------------------------------------------------------
# bs=32 is fine for resnet on a T4
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)

resnet = build_pretrained('resnet101', 7).to(device)
resnet, hist_resnet = finetune(resnet, 'resnet101', trl, val_loader, 'multi')

res = evaluate(resnet, te_loader, 'multi')
print()
print("TEST resnet101 (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'resnet101 / multi')

torch.save({'state_dict': resnet.state_dict(), 'history': hist_resnet,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'resnet101_multi.pt'))
print("saved", os.path.join(OUT, 'resnet101_multi.pt'))

In [ ]:
# ---- 12. densenet121, 7 classes ----------------------------------------------------
# densenet121 was the lightest of the four (6.96M) and it fits bs=32 easily
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)

densenet = build_pretrained('densenet121', 7).to(device)
densenet, hist_densenet = finetune(densenet, 'densenet121', trl, val_loader, 'multi')

res = evaluate(densenet, te_loader, 'multi')
print()
print("TEST densenet121 (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'densenet121 / multi')

torch.save({'state_dict': densenet.state_dict(), 'history': hist_densenet,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'densenet121_multi.pt'))
print("saved", os.path.join(OUT, 'densenet121_multi.pt'))

In [ ]:
# ---- 13. efficientnet_b3, 7 classes ------------------------------------------------
# it expects 300px normally but with 224 it still works and is much faster
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=32)

effnet = build_pretrained('efficientnet_b3', 7).to(device)
effnet, hist_effnet = finetune(effnet, 'efficientnet_b3', trl, val_loader, 'multi')

res = evaluate(effnet, te_loader, 'multi')
print()
print("TEST efficientnet_b3 (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'efficientnet_b3 / multi')

torch.save({'state_dict': effnet.state_dict(), 'history': hist_effnet,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'efficientnet_b3_multi.pt'))
print("saved", os.path.join(OUT, 'efficientnet_b3_multi.pt'))

In [ ]:
# ---- 14. convnext_base, 7 classes --------------------------------------------------
# 87.6M params, bs=32 does not fit on a free T4, so bs=16 and we use a
# smaller lr for the full unfreeze (with 1e-3 the loss went to NaN once)
set_seed()
trl, val_loader, te_loader = make_loaders(mode='multi', bs=16)

convnext = build_pretrained('convnext_base', 7).to(device)
convnext, hist_convnext = finetune(convnext, 'convnext_base', trl, val_loader, 'multi')

res = evaluate(convnext, te_loader, 'multi')
print()
print("TEST convnext_base (multi)  acc %.4f  macro-F1 %.4f" % (res['acc'], res['macro_f1']))
_print_class_report(res, 'convnext_base / multi')

torch.save({'state_dict': convnext.state_dict(), 'history': hist_convnext,
            'test_acc': res['acc'], 'test_macro_f1': res['macro_f1']},
           os.path.join(OUT, 'convnext_base_multi.pt'))
print("saved", os.path.join(OUT, 'convnext_base_multi.pt'))

In [ ]:
# ---- 15. binary task: the loss was wrong at first ----------------------------------
# the first binary runs used a plain BCEWithLogitsLoss. AUC looked fine (~0.85)
# but F1 was ~0.45: only 10.8% of train images are melanoma so the loss is
# happy with "never positive".
#
# the invalid first run, kept for the report (column F1@0.5):
#
#   tl_b, vl_b, te_b = make_loaders(mode='binary', bs=32, use_sampler=False)
#   m = build_pretrained('resnet101', 1).to(device)
#   m, _ = finetune(m, 'resnet101-bin-old', tl_b, vl_b, 'binary')
#   r = evaluate(m, te_b, 'binary')
#   print(r['auc'], r['f1'])        # auc 0.88, f1 0.46
#
# fix: weight the positive class with neg/pos (POS_WEIGHT from cell 1, ~8.09).
# we also tried sampler + pos_weight together: too much, precision dropped to
# ~0.3 because the model started saying melanoma everywhere.
POS_WEIGHT = float(POS_WEIGHT)     # from the split cell

def binary_loss():
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))

print("binary loss = BCEWithLogitsLoss(pos_weight=%.3f)" % POS_WEIGHT)

In [ ]:
# ---- 16. binary: resnet101 ---------------------------------------------------------
# 9 epochs in total (2 + 4 + 8 with early stop), the val F1 is noisy here
# because only ~160 images in val are melanoma
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=32, use_sampler=False)

resnet_bin = build_pretrained('resnet101', 1).to(device)
resnet_bin, hist_resnet_bin = finetune(resnet_bin, 'resnet101-bin', trl_b, val_b, 'binary',
                                       loss_fn=binary_loss())

res_b = evaluate(resnet_bin, te_b, 'binary')
print()
print("TEST resnet101 (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': resnet_bin.state_dict(), 'history': hist_resnet_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'resnet101_binary.pt'))
print("saved", os.path.join(OUT, 'resnet101_binary.pt'))

In [ ]:
# ---- 17. binary: densenet121 -------------------------------------------------------
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=32, use_sampler=False)

densenet_bin = build_pretrained('densenet121', 1).to(device)
densenet_bin, hist_densenet_bin = finetune(densenet_bin, 'densenet121-bin', trl_b, val_b, 'binary',
                                           loss_fn=binary_loss())

res_b = evaluate(densenet_bin, te_b, 'binary')
print()
print("TEST densenet121 (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': densenet_bin.state_dict(), 'history': hist_densenet_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'densenet121_binary.pt'))
print("saved", os.path.join(OUT, 'densenet121_binary.pt'))

In [ ]:
# ---- 18. binary: efficientnet_b3 ---------------------------------------------------
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=32, use_sampler=False)

effnet_bin = build_pretrained('efficientnet_b3', 1).to(device)
effnet_bin, hist_effnet_bin = finetune(effnet_bin, 'efficientnet_b3-bin', trl_b, val_b, 'binary',
                                       loss_fn=binary_loss())

res_b = evaluate(effnet_bin, te_b, 'binary')
print()
print("TEST efficientnet_b3 (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': effnet_bin.state_dict(), 'history': hist_effnet_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'efficientnet_b3_binary.pt'))
print("saved", os.path.join(OUT, 'efficientnet_b3_binary.pt'))

In [ ]:
# ---- 19. binary: convnext_base -----------------------------------------------------
# bs=16 again or the T4 runs out of memory
set_seed()
trl_b, val_b, te_b = make_loaders(mode='binary', bs=16, use_sampler=False)

convnext_bin = build_pretrained('convnext_base', 1).to(device)
convnext_bin, hist_convnext_bin = finetune(convnext_bin, 'convnext_base-bin', trl_b, val_b, 'binary',
                                           loss_fn=binary_loss())

res_b = evaluate(convnext_bin, te_b, 'binary')
print()
print("TEST convnext_base (binary)  acc %.4f  auc %.4f  f1@0.5 %.4f"
      % (res_b['acc'], res_b['auc'], res_b['f1']))

torch.save({'state_dict': convnext_bin.state_dict(), 'history': hist_convnext_bin,
            'test_acc': res_b['acc'], 'test_auc': res_b['auc'], 'test_f1': res_b['f1']},
           os.path.join(OUT, 'convnext_base_binary.pt'))
print("saved", os.path.join(OUT, 'convnext_base_binary.pt'))

In [ ]:
# ---- 21. test predictions for all 10 checkpoints -----------------------------------
# putting them in one dict so the table cells do not have to reload the
# models again (we did that first, it took 20 min on the CPU)
PRED_PATH = os.path.join(OUT, 'predictions.pt')

# the timm model name per checkpoint. we tried to cut it out of the filename
# at first but efficientnet_b3_multi.pt -> 'efficientnet' does not exist
TIMM_FOR = {'custom_multi.pt': None, 'custom_binary.pt': None,
            'resnet101_multi.pt': 'resnet101', 'resnet101_binary.pt': 'resnet101',
            'densenet121_multi.pt': 'densenet121', 'densenet121_binary.pt': 'densenet121',
            'efficientnet_b3_multi.pt': 'efficientnet_b3', 'efficientnet_b3_binary.pt': 'efficientnet_b3',
            'convnext_base_multi.pt': 'convnext_base', 'convnext_base_binary.pt': 'convnext_base'}

predictions = {}
for disp, fname, n_out, task in MODEL_FILES:
    path = os.path.join(OUT, fname)
    if not os.path.exists(path):
        print("skip, no file:", fname)
        continue
    timm_name = TIMM_FOR[fname]
    if timm_name is None:
        model = CustomCNN(n_out=n_out)
    else:
        # pretrained=False, we are loading our own weights from the checkpoint
        model = timm.create_model(timm_name, pretrained=False, num_classes=n_out)
    # the checkpoints were written on the GPU, map to cpu first so this cell
    # also runs after a runtime restart
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'])
    model = model.to(device)

    _, _, te = make_loaders(mode=task, bs=32, use_sampler=False)
    r = evaluate(model, te, task)
    predictions[(disp, task)] = {'y': r['y'], 'p': r['p']}
    print("%-16s %-7s acc %.4f" % (disp, task, r['acc']))
    del model

torch.save(predictions, PRED_PATH)
print("saved", PRED_PATH)

In [ ]:
# ---- 22. table 1 (binary) and table 2 (multi) --------------------------------------
from sklearn.metrics import roc_auc_score, average_precision_score

ORDER = ['Custom CNN', 'ResNet-101', 'DenseNet-121', 'EfficientNet-B3', 'ConvNeXt-Base']

def tune_threshold(y, p, lo=0.05, hi=0.95, step=0.01):
    """brute force on a grid, returns the threshold with the best F1.
    NOTE: this must be called with the VALIDATION predictions (see cell 23)."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(lo, hi, step):
        pred = (p >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

rows1, rows2 = [], []
for disp in ORDER:
    # binary
    y = predictions[(disp, 'binary')]['y']
    p = predictions[(disp, 'binary')]['p']
    rows1.append({'model': disp,
                  'acc': accuracy_score(y, (p >= 0.5).astype(int)),
                  'auc': roc_auc_score(y, p),
                  'pr_auc': average_precision_score(y, p),
                  'f1_05': precision_recall_fscore_support(y, (p >= 0.5).astype(int), average='binary', zero_division=0)[2]})
    # multi
    y = predictions[(disp, 'multi')]['y']
    p = predictions[(disp, 'multi')]['p']
    pred = p.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    rows2.append({'model': disp, 'acc': accuracy_score(y, pred), 'macro_p': pr, 'macro_r': rc, 'macro_f1': f1})

t1 = pd.DataFrame(rows1)
t2 = pd.DataFrame(rows2)

print("TABLE 1 - binary (melanoma vs rest), test set")
print(t1.round(4).to_string(index=False))
print()
print("TABLE 2 - 7 classes, test set")
print(t2.round(4).to_string(index=False))

t1.to_csv(os.path.join(OUT, 'table1_binary.csv'), index=False)
t2.to_csv(os.path.join(OUT, 'table2_multi.csv'), index=False)
print()
print("tables saved to", OUT)

In [ ]:
# ---- 23. binary threshold tuned on the VALIDATION set ------------------------------
# v1 tuned the threshold on the test set (it is in the first version of the
# report, section 4, "limitations"). that is leakage: we now take the models
# again, predict the val set, pick the threshold there and apply it once.
thr_rows = []
for disp, fname, n_out, task in MODEL_FILES:
    if task != 'binary':
        continue
    path = os.path.join(OUT, fname)
    timm_name = TIMM_FOR[fname]
    model = CustomCNN(n_out=1) if timm_name is None else timm.create_model(timm_name, pretrained=False, num_classes=1)
    model.load_state_dict(torch.load(path, map_location='cpu')['state_dict'])
    model = model.to(device)

    _, va_b, te_b = make_loaders(mode='binary', bs=32, use_sampler=False)
    rv = evaluate(model, va_b, 'binary')
    t_star, f1_val = tune_threshold(rv['y'], rv['p'])

    y = predictions[(disp, 'binary')]['y']
    p = predictions[(disp, 'binary')]['p']
    pred = (p >= t_star).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
    thr_rows.append({'model': disp, 'thr_val': t_star, 'f1_val': f1_val,
                     'precision': pr, 'recall': rc, 'f1_test': f1})
    print("%-16s thr %.2f  val F1 %.4f  ->  test F1 %.4f" % (disp, t_star, f1_val, f1))
    del model

t1b = pd.DataFrame(thr_rows)
print()
print("TABLE 1b - binary, threshold chosen on val, evaluated on test")
print(t1b.round(4).to_string(index=False))
t1b.to_csv(os.path.join(OUT, 'table1b_binary_threshold.csv'), index=False)

In [ ]:
# ---- 24. per class metrics for the best multi models -------------------------------
pr, rc, f1, sup = precision_recall_fscore_support(
    predictions[('ConvNeXt-Base', 'multi')]['y'],
    predictions[('ConvNeXt-Base', 'multi')]['p'].argmax(1),
    labels=range(7), zero_division=0)
t3 = pd.DataFrame({'class': CLASSES, 'precision': pr, 'recall': rc, 'f1': f1, 'n_test': sup})
print("TABLE 3 - per class, ConvNeXt-Base")
print(t3.round(4).to_string(index=False))
t3.to_csv(os.path.join(OUT, 'table3_perclass_convnext.csv'), index=False)

cc = confusion_matrix(predictions[('ConvNeXt-Base', 'multi')]['y'],
                      predictions[('ConvNeXt-Base', 'multi')]['p'].argmax(1))
print()
print("confusion matrix (rows = true)")
print(pd.DataFrame(cc, index=CLASSES, columns=CLASSES).to_string())

In [ ]:
# ---- 25. figures -------------------------------------------------------------------
def savefig(fig, name):
    p = os.path.join(OUT, name)
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    print("saved", p)

# fig 1 - ROC curves, binary
plt.figure(figsize=(6, 5))
for disp in ORDER:
    y = predictions[(disp, 'binary')]['y']
    p = predictions[(disp, 'binary')]['p']
    fpr, tpr, _ = roc_curve(y, p)
    plt.plot(fpr, tpr, label='%s (AUC %.3f)' % (disp, roc_auc_score(y, p)))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('binary: melanoma vs non-melanoma')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
savefig(plt.gcf(), 'fig1_roc_binary.png')

# fig 2 - the two best multi models, confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, disp in zip(axes, ['ConvNeXt-Base', 'DenseNet-121']):
    cm = confusion_matrix(predictions[(disp, 'multi')]['y'],
                          predictions[(disp, 'multi')]['p'].argmax(1))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(7)); ax.set_xticklabels(CLASSES, rotation=45)
    ax.set_yticks(range(7)); ax.set_yticklabels(CLASSES)
    ax.set_title(disp); ax.set_xlabel('predicted'); ax.set_ylabel('true')
    for i in range(7):
        for j in range(7):
            if cm[i, j] > 0:
                ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                        color='white' if cm[i, j] > cm.max() / 2 else 'black')
savefig(fig, 'fig2_confusion_matrices.png')
plt.close(fig)

# fig 3 - training curves, custom cnn vs convnext (multi)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in [(axes[0], 'train_loss', 'train loss'), (axes[1], 'val_score', 'val macro-F1')]:
    ax.plot(hist_custom_multi[key], label='Custom CNN')
    ax.plot(hist_convnext[key], label='ConvNeXt-Base')
    ax.set_xlabel('epoch'); ax.set_ylabel(title); ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.3)
savefig(fig, 'fig3_training_curves.png')
plt.close(fig)

In [ ]:
# ---- 26. parameters + speed on the CPU ---------------------------------------------
# the free T4 ran out of quota near the deadline, so the speed numbers are
# measured on the colab CPU (it is slow, 44 s per iteration for convnext)
import time

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

def cpu_speed(m, bs=16, warmup=3, iters=8):
    m = m.to('cpu').eval()
    x = torch.randn(bs, 3, 224, 224)
    with torch.no_grad():
        for _ in range(warmup):
            m(x)
        t0 = time.time()
        for _ in range(iters):
            m(x)
        dt = time.time() - t0
    return bs * iters / dt

models_for_speed = {
    'Custom CNN':      CustomCNN(7),
    'ResNet-101':      timm.create_model('resnet101', pretrained=False, num_classes=7),
    'DenseNet-121':    timm.create_model('densenet121', pretrained=False, num_classes=7),
    'EfficientNet-B3': timm.create_model('efficientnet_b3', pretrained=False, num_classes=7),
    'ConvNeXt-Base':   timm.create_model('convnext_base', pretrained=False, num_classes=7),
}

rows4 = []
for name, m in models_for_speed.items():
    p = count_params(m)
    s = cpu_speed(m)
    rows4.append({'model': name, 'params_M': round(p, 2), 'img_per_sec_cpu': round(s, 2)})
    print("%-16s %6.2fM  %6.2f img/s" % (name, p, s))

t4 = pd.DataFrame(rows4)
print()
print("TABLE 4 - params and CPU throughput")
print(t4.to_string(index=False))
t4.to_csv(os.path.join(OUT, 'table4_params_speed.csv'), index=False)

In [ ]:
# ---- 27. bootstrap 95% CI for macro-F1 (multi) -------------------------------------
# the test set has only 7 'df' images, so one lucky image moves the macro F1
# by ~0.03. the CI shows whether the gap between two models is real.
def bootstrap_macro_f1(y, p, n=500, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 7:      # a resample without all 7 classes gives a fake F1
            continue
        _, _, f1, _ = precision_recall_fscore_support(y[idx],
                                                      p[idx].argmax(1),
                                                      average='macro', zero_division=0)
        scores.append(f1)
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return float(np.mean(scores)), float(lo), float(hi)

print("macro-F1 with 95% bootstrap CI (500 resamples)")
ci_rows = []
for disp in ORDER:
    y = predictions[(disp, 'multi')]['y']
    p = predictions[(disp, 'multi')]['p']
    mean, lo, hi = bootstrap_macro_f1(y, p)
    print("  %-16s %.4f  [%.4f, %.4f]" % (disp, mean, lo, hi))
    ci_rows.append({'model': disp, 'macro_f1': round(mean, 4),
                    'ci_low': round(lo, 4), 'ci_high': round(hi, 4)})

pd.DataFrame(ci_rows).to_csv(os.path.join(OUT, 'table2_ci.csv'), index=False)
print("saved table2_ci.csv")

In [ ]:
# ---- 28. appendix: correct and wrong examples per class ----------------------------
# appendix grid for the report, 3 correct + 3 wrong per class, convnext_base
pred_cx = predictions[('ConvNeXt-Base', 'multi')]['p'].argmax(1)
y_cx    = predictions[('ConvNeXt-Base', 'multi')]['y']

fig, axes = plt.subplots(7, 6, figsize=(13, 16))
for ci in range(7):
    idx = np.where(y_cx == ci)[0]
    good = [i for i in idx if pred_cx[i] == ci][:3]
    bad  = [i for i in idx if pred_cx[i] != ci][:3]

    for j, i in enumerate(good):
        img = cv2.cvtColor(cv2.imread(test_df.iloc[i]['path']), cv2.COLOR_BGR2RGB)
        axes[ci, j].imshow(img)
        axes[ci, j].set_title('OK ' + CLASSES[ci], fontsize=8)
        axes[ci, j].axis('off')
    for j, i in enumerate(bad):
        img = cv2.cvtColor(cv2.imread(test_df.iloc[i]['path']), cv2.COLOR_BGR2RGB)
        axes[ci, 3 + j].imshow(img)
        axes[ci, 3 + j].set_title('%s->%s' % (CLASSES[ci], CLASSES[pred_cx[i]]), fontsize=8)
        axes[ci, 3 + j].axis('off')
    for j in range(len(good), 3):
        axes[ci, j].axis('off')
    for j in range(len(bad), 3):
        axes[ci, 3 + j].axis('off')

plt.tight_layout()
savefig(fig, 'fig_appendix_examples.png')
plt.close(fig)